## Notebook30

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
rva = pl.read_csv(ub + "data/flightsrva_flights.csv.gz", null_values=["NA"])
airline = pl.read_csv(ub + "data/flightsrva_airlines.csv.gz", null_values=["NA"])
airport = pl.read_csv(ub + "data/flightsrva_airports.csv.gz", null_values=["NA"])
plane = pl.read_csv(ub + "data/flightsrva_planes.csv.gz", null_values=["NA"])

### Questions

This notebook is a review of the whole first semester: types, filtering and
sorting, plotting, grouping, windows, joins, and reshaping. Nothing here is
new. The dataset is four tables about commercial flights out of Richmond
(RIC) in 2019. `rva` is the main table, one row per flight, with scheduled
and actual times, delays in minutes, the carrier, the tail number of the
aircraft, the destination, and the distance flown. `airline` maps a
two-letter `carrier` code to a full company name. `airport` maps a
three-letter FAA code to a name, a location, and a time zone; `rva`'s
`dest` column matches `airport`'s `faa` column, not its own name. `plane`
is a registry of individual aircraft, keyed by `tailnum`, with the
manufacturer, model, and seat count. Print results unless a question says
otherwise.

1. Sort `rva` by distance, longest flight first. Select just `carrier`,
`dest`, and `distance`, and print the top 5.

The top 5 are all the same flight, UA to Denver. Distance belongs to the
route, not the individual flight, so every flight on a route ties.

2. Filter to flights headed to Atlanta (`ATL`) or Chicago O'Hare (`ORD`)
using `.is_in()`. Print how many there are with `.select(pl.len())`.

3. Rename `dep_delay` to `delay_departure` and `arr_delay` to
`delay_arrival`. Print `carrier` and the two renamed columns for the first
5 rows.

4. How many distinct aircraft, by `tailnum`, flew out of Richmond in 2019?
Answer it two ways: once with `.select(c.tailnum.n_unique())`, and once
with `.unique(subset="tailnum", keep="first")` followed by
`.select(pl.len())`. Confirm the two numbers agree.

5. Add a `delay_total` column (`dep_delay` plus `arr_delay`), and a
`status` column built with a chained `pl.when().then().otherwise()`:
`"cancelled"` when `dep_delay` is null, `"delayed"` when `dep_delay` is
over 15, and `"on time"` otherwise. Print `carrier`, `dep_delay`,
`delay_total`, and `status` for 5 rows, then count how many flights fall
into each status.

6. Print `rva.schema`. Then count the nulls in `dep_time` and in
`dep_delay` and check that the two counts match.

The counts match because both columns go missing for the same reason: a
cancelled flight was never scheduled to depart, so it has no actual
departure time and no delay to measure.

7. Uppercase every `carrier` code in `rva` with `.str.to_uppercase()` and
print the distinct values (they should look unchanged, since the codes are
already uppercase). Then, in `plane`, use `.str.contains()` to filter to
aircraft whose `manufacturer` contains `"BOEING"`, and print how many
distinct `model` values come back.

8. Plot `dep_delay` against `arr_delay` with `geom_point()`, using
`alpha=0.1` so the overplotting is readable, and add a `geom_smooth()`
trend line with `method="lm"` and `se=False`.

9. Count flights per carrier, sort ascending, and plot as a bar chart with
`geom_col()` and `coord_flip()`.

10. Build a `date` column with `pl.date(c.year, c.month, c.day)`. Group by
`date`, count the flights on each day, sort by date, and plot the daily
count with `geom_line()`.

11. Group by `carrier` and compute the mean and median `dep_delay`, plus a
count, sorted by the mean, worst first.

Every median is negative while several means are well above zero. Most
flights leave a little early or on time, and a smaller number of very late
flights pull the mean up without moving the median much.

12. Group by `carrier` and `month` together. In each group, count the
flights and count how many had `dep_delay` over 15. Add a `prop_delayed`
column (the second count divided by the first) and sort by it, worst
first.

13. Without any grouping, use `.select()` with aggregating expressions to
find the minimum, mean, and maximum `distance`, the 90th percentile of
`dep_delay`, and the correlation between `dep_delay` and `arr_delay`
(`pl.corr()`).

14. Cancelled flights have no delay to speak of, so drop the rows where
`dep_delay` is null first. Then add a `delay_dev` column: each flight's
`dep_delay` minus its own carrier's mean `dep_delay`, computed with
`.over()`. Sort descending and print the 5 flights that ran latest
relative to their own carrier's usual performance.

15. Sort `rva` by `dep_delay`, descending, and print the top 5. Then
explain what went wrong.

**Answer**:


The worst flight of the year left almost 30 hours late.

16. Build the same daily table as question 10, but keep `dep_delay`'s mean
as well as the count. Sort by date, then add `n_cum` (`.cum_sum()` on the
daily count) and `delay_roll` (a 7-day `.rolling_mean()` of the daily mean
delay). Plot `delay_roll` against `date` with `geom_line()`.

17. Plot a histogram of `dep_delay` with `geom_histogram()`, `binwidth=10`
and `boundary=0`.

18. Plot `dep_delay` by `carrier` with `geom_boxplot()`. The worst outliers
stretch the y-axis until the boxes themselves are unreadable, so crop the
view with `.coord_cartesian(ylim=(-30, 60))` rather than filtering the
data or setting scale limits, both of which would throw out the extreme
flights before the boxes are even computed.

19. Facet the same histogram from question 17 by `carrier` with
`.facet_wrap("carrier")`.

20. Join `rva` to `airline` on `carrier`, and print `carrier`, `name`, and
`dest` for 5 rows.

21. Before joining to `airport`, check that `faa` is actually a key there
with `.select(c.faa.n_unique(), pl.len())`. Then join `rva` to `airport`
with `left_on="dest"` and `right_on="faa"`, group by the airport `name`,
count the flights to each, and print the 10 busiest destinations.

22. Use `how="anti"` to join `rva` to `plane` on `tailnum` and find the
flights whose aircraft has no matching row in the registry. Print how many
flights and how many distinct tail numbers that is.

23. Drop cancelled flights, then `.unpivot()` `dep_delay` and `arr_delay`
into a `delay_type`/`minutes` pair, keeping `carrier` as the index. Plot
`minutes` by `delay_type` with `geom_boxplot()`, cropped the same way as
question 18. Then group the long table by `carrier` and `delay_type`,
take the mean `minutes`, and `.pivot()` it back to one row per carrier
with a column for each delay type.